# 02 Preprocessing: add synthetic prices

ABO listings have no price field. This notebook adds **simulated (synthetic) prices** so price filtering (e.g. 'under $50') can be demonstrated. They are not real prices.

Prerequisites: `01_data_exploration.ipynb` saved `listings_en_raw.parquet` to Drive, and `src/preprocess/synthetic_prices.py` is committed to the repo.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/shoptalk'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/anbuaru/shoptalk.git'
    except Exception:
        url = 'https://github.com/anbuaru/shoptalk.git'
    !git clone {url} {REPO}
else:
    !git -C {REPO} pull

%cd /content/shoptalk
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/shoptalk
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
import sys
import os
import pandas as pd

sys.path.append(REPO)
assert os.path.exists(f'{REPO}/preprocess/synthetic_prices.py'), 'Commit preprocess/synthetic_prices.py to GitHub first'
from preprocess.synthetic_prices import synthetic_price

PROJECT_DIR = '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk'
df = pd.read_parquet(f'{PROJECT_DIR}/data/listings_en_raw.parquet')
print(df.shape)
df.head(3)

(147702, 10)


,item_id,name_en,brand_en,color_en,ptype,bullets_en,desc_en,keywords_en,main_image_id,country
0,B06X9STHNG,None,None,None,SHOES,,,,81iZlv3bjpL,NL
1,B07P8ML82R,None,None,None,HARDWARE,,,,619y9YG9cnL,MX
2,B07H9GMYXS,"AmazonBasics PETG 3D Printer Filament, 1.75mm,...",AmazonBasics,Translucent Yellow,MECHANICAL_COMPONENTS,3D printer filament with 1.75mm diameter + / -...,,"3d printer filament, petg printer filament, pe...",81NP7qh2L6L,AE


In [3]:
df['price'] = [synthetic_price(i, t) for i, t in zip(df['item_id'], df['ptype'])]
df['price_currency'] = 'USD'
df['price_is_synthetic'] = True
df[['item_id', 'ptype', 'price', 'price_is_synthetic']].head(10)

,item_id,ptype,price,price_is_synthetic
0,B06X9STHNG,SHOES,54.99,True
1,B07P8ML82R,HARDWARE,44.99,True
2,B07H9GMYXS,MECHANICAL_COMPONENTS,32.99,True
3,B07CTPR73M,SOFA,500.99,True
4,B01MTEI8M6,SHOES,38.99,True
5,B0853X2F4M,CELLULAR_PHONE_CASE,9.99,True
6,B07R91S92W,CELLULAR_PHONE_CASE,9.99,True
7,B0896LJNLH,DRINKING_CUP,81.99,True
8,B0853WVZ6Q,CELLULAR_PHONE_CASE,24.99,True
9,B07R51S57D,FURNITURE_COVER,129.99,True


#Sanity-check the prices

In [4]:
print(df['price'].describe())

med = df.groupby('ptype')['price'].median().sort_values()
print('Cheapest types by median price:')
print(med.head(10))
print('Most expensive types by median price:')
print(med.tail(10))

under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

# Determinism check: running the generator again must give identical prices
again = [synthetic_price(i, t) for i, t in zip(df['item_id'].head(1000), df['ptype'].head(1000))]
assert again == df['price'].head(1000).tolist(), 'Prices are not deterministic'
print('Determinism check passed')

count    147702.000000
mean         43.377239
std          81.652684
min           1.990000
25%          13.990000
50%          23.990000
75%          47.990000
max        2345.990000
Name: price, dtype: float64
Cheapest types by median price:
ptype
AIR_PURIFIER                    10.99
VEHICLE_SAFETY_CAMERA           11.99
GROCERY                         11.99
CARRYING_CASE_OR_BAG            12.99
CE_CARRYING_CASE_OR_BAG         13.99
KICK_SCOOTER                    13.99
MEMORY_READER                   13.99
CELLULAR_PHONE_CASE             13.99
COSMETIC_CASE                   14.99
AMAZON_BOOK_READER_ACCESSORY    14.99
Name: price, dtype: float64
Most expensive types by median price:
ptype
OUTBUILDING           128.99
MECHANICAL_LIGHTER    150.99
TABLE                 184.49
SPORT_TABLE_GAME      204.99
DESK                  204.99
MATTRESS              365.99
BED                   420.99
BED_FRAME             420.99
AIR_MATTRESS          480.99
SOFA                  660.99
Name: pr

In [5]:
out = f'{PROJECT_DIR}/data/listings_en_priced.parquet'
df.to_parquet(out)
print('Saved', out, df.shape)

Saved /content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/listings_en_priced.parquet (147702, 13)


In [6]:
print(df['price'].describe())
print(df.groupby('ptype')['price'].median().sort_values().head(10))
print(df.groupby('ptype')['price'].median().sort_values().tail(10))

# Does the demo query have results?
under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

count    147702.000000
mean         43.377239
std          81.652684
min           1.990000
25%          13.990000
50%          23.990000
75%          47.990000
max        2345.990000
Name: price, dtype: float64
ptype
AIR_PURIFIER                    10.99
VEHICLE_SAFETY_CAMERA           11.99
GROCERY                         11.99
CARRYING_CASE_OR_BAG            12.99
CE_CARRYING_CASE_OR_BAG         13.99
KICK_SCOOTER                    13.99
MEMORY_READER                   13.99
CELLULAR_PHONE_CASE             13.99
COSMETIC_CASE                   14.99
AMAZON_BOOK_READER_ACCESSORY    14.99
Name: price, dtype: float64
ptype
OUTBUILDING           128.99
MECHANICAL_LIGHTER    150.99
TABLE                 184.49
SPORT_TABLE_GAME      204.99
DESK                  204.99
MATTRESS              365.99
BED                   420.99
BED_FRAME             420.99
AIR_MATTRESS          480.99
SOFA                  660.99
Name: price, dtype: float64
112955 items under $50
ptype
CELLULAR_PHONE_CASE  

In [7]:
!git pull

Already up to date.


In [8]:
for t in ['SHIRT', 'BOOT', 'SHOES', 'SANDAL', 'WALLET', 'BAG', 'LAMP', 'SOFA', 'CHAIR', 'RING', 'WATCH', 'COFFEE']:
    print(f'{t:8s}', df['ptype'].str.contains(t, na=False).sum())
print(df['ptype'].value_counts().head(40))

SHIRT    0
BOOT     2009
SHOES    12965
SANDAL   1845
WALLET   176
BAG      1324
LAMP     691
SOFA     1199
CHAIR    2173
RING     3859
WATCH    112
COFFEE   313
ptype
CELLULAR_PHONE_CASE           64853
SHOES                         12965
GROCERY                        6546
HOME                           5264
HOME_BED_AND_BATH              3082
HOME_FURNITURE_AND_DECOR       2255
CHAIR                          2100
BOOT                           2009
SANDAL                         1845
FINERING                       1540
HEALTH_PERSONAL_CARE           1449
FINENECKLACEBRACELETANKLET     1377
ACCESSORY                      1362
SOFA                           1199
OFFICE_PRODUCTS                1152
FINEEARRING                    1137
PET_SUPPLIES                   1064
SPORTING_GOODS                  972
TABLE                           936
HARDWARE_HANDLE                 860
RUG                             817
HANDBAG                         792
LIGHT_BULB                      756
KITC

##There are no shirts at all (SHIRT is 0), so the “red shirt for men” example has to go, and apparel here means shoes, boots, sandals, hats, handbags, and jewelry. Shoes (12,965 items, 9%) and sandals are the apparel strength. The rest of the catalog is mostly home and furniture, which suits “cozy living room” and gift-style queries. Phone cases are still 44% of the data and will be mostly dropped.